# NB8 — Evidence-Gated GNN & Baseline Benchmarking

**Purpose:** Train an Evidence-Gated Graph Neural Network on the PCBS-annotated
PPI graph from NB7, benchmark against five baselines, and run ablation experiments
to quantify each component's contribution.

| Input | Source | File |
|-------|--------|------|
| Edge list | NB7 | `results/ppi_edge_list.csv` |
| Node features | NB7 | `results/ppi_node_features.csv` |
| PCBS scores | NB6 | `results/pcbs_scores.csv` |

**Models:**
1. PPR (Personalised PageRank) — topology-only baseline
2. GCN (Graph Convolutional Network)
3. GraphSAGE
4. GAT (Graph Attention Network)
5. GIN (Graph Isomorphism Network)
6. **EG-GNN** (Evidence-Gated GNN — ours)

**Primary Metrics:** AUPRC, Precision@K, Recall@K, NDCG  
**Complementary:** AUROC

**Task:** Bridge gene ranking — predict which genes are high-PCBS bridge candidates
using graph structure + node features.

**Output:** `results/NB8_GNN_results.zip`

In [ ]:
!pip install torch-geometric torch-scatter torch-sparse -q 2>/dev/null
# If the above fails on Colab, use:
# !pip install torch-geometric -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import os, warnings, zipfile, io
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F

os.makedirs('results', exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'NB8 — Evidence-Gated GNN & Benchmarking')
print(f'Device: {device}')
print('=' * 45)

## 1 · Load Graph Data

In [ ]:
from google.colab import files

print("Upload NB6 + NB7 results ZIPs\n")

uploaded = files.upload()

for fname, content in uploaded.items():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(io.BytesIO(content), 'r') as zf:
            zf.extractall('.')
            print(f"Extracted {fname}: {len(zf.namelist())} files")

edges_df = pd.read_csv('results/ppi_edge_list.csv')
nodes_df = pd.read_csv('results/ppi_node_features.csv')
pcbs_df  = pd.read_csv('results/pcbs_scores.csv')

print(f'\nEdges: {len(edges_df):,}')
print(f'Nodes: {len(nodes_df):,}')
print(f'PCBS scored genes: {len(pcbs_df):,}')

## 2 · Build PyG Data Object

In [ ]:
from torch_geometric.data import Data

# ---- Gene-to-index mapping ----
all_genes = sorted(nodes_df['Gene'].unique())
gene2idx = {g: i for i, g in enumerate(all_genes)}
idx2gene = {i: g for g, i in gene2idx.items()}
num_nodes = len(all_genes)

# ---- Node features: P_g, T_g, C_g, D_g, K_g (5 features) ----
# We exclude PCBS itself to avoid label leakage — PCBS is the TARGET
feature_cols = ['P_g', 'T_g', 'C_g', 'D_g', 'K_g']
nodes_indexed = nodes_df.set_index('Gene')

X = np.zeros((num_nodes, len(feature_cols)), dtype=np.float32)
for i, gene in enumerate(all_genes):
    if gene in nodes_indexed.index:
        X[i] = nodes_indexed.loc[gene, feature_cols].values

x = torch.tensor(X, dtype=torch.float)

# ---- Target: PCBS score (regression) ----
pcbs_indexed = pcbs_df.set_index('Gene')['PCBS']
y_vals = np.array([pcbs_indexed.get(g, 0.0) for g in all_genes], dtype=np.float32)
y = torch.tensor(y_vals, dtype=torch.float)

# ---- Edge index (undirected → both directions) ----
src, dst, edge_w = [], [], []
for _, row in edges_df.iterrows():
    g1, g2 = row['gene1'], row['gene2']
    if g1 in gene2idx and g2 in gene2idx:
        i, j = gene2idx[g1], gene2idx[g2]
        src.extend([i, j])
        dst.extend([j, i])
        w = row['weight']
        edge_w.extend([w, w])

edge_index = torch.tensor([src, dst], dtype=torch.long)
edge_weight = torch.tensor(edge_w, dtype=torch.float)

data = Data(x=x, edge_index=edge_index, edge_attr=edge_weight, y=y)
data = data.to(device)

print(f'PyG Data object:')
print(f'  Nodes: {data.num_nodes:,}')
print(f'  Edges: {data.num_edges:,} (directed, i.e. {data.num_edges//2:,} undirected)')
print(f'  Node features: {data.num_node_features}')
print(f'  Target range: [{y.min():.4f}, {y.max():.4f}]')

## 3 · Train / Validation / Test Split

Split nodes into 60/20/20 train/val/test. Stratify by PCBS quantile to
ensure each split has high-scoring bridge genes.

In [ ]:
from sklearn.model_selection import train_test_split

# Stratify by PCBS bins
pcbs_np = y.cpu().numpy()
bins = pd.qcut(pcbs_np, q=5, labels=False, duplicates='drop')

idx_all = np.arange(num_nodes)
idx_train, idx_temp, _, bins_temp = train_test_split(
    idx_all, bins, test_size=0.4, random_state=42, stratify=bins)
idx_val, idx_test = train_test_split(
    idx_temp, test_size=0.5, random_state=42, stratify=bins_temp)

train_mask = torch.zeros(num_nodes, dtype=torch.bool)
val_mask   = torch.zeros(num_nodes, dtype=torch.bool)
test_mask  = torch.zeros(num_nodes, dtype=torch.bool)
train_mask[idx_train] = True
val_mask[idx_val] = True
test_mask[idx_test] = True

data.train_mask = train_mask.to(device)
data.val_mask   = val_mask.to(device)
data.test_mask  = test_mask.to(device)

print(f'Train: {train_mask.sum():,}  Val: {val_mask.sum():,}  Test: {test_mask.sum():,}')
for name, mask in [('Train', train_mask), ('Val', val_mask), ('Test', test_mask)]:
    vals = pcbs_np[mask.numpy()]
    print(f'  {name} — mean PCBS: {vals.mean():.4f}, '
          f'top-5% genes: {(vals >= np.percentile(pcbs_np, 95)).sum()}')

## 4 · Model Definitions

### 4a — Baseline GNN Models

In [ ]:
from torch_geometric.nn import GCNConv, SAGEConv, GATConv, GINConv


class GCN(nn.Module):
    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden)
        self.conv2 = GCNConv(hidden, hidden)
        self.head = nn.Linear(hidden, 1)

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index, data.edge_attr))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.relu(self.conv2(x, data.edge_index, data.edge_attr))
        return self.head(x).squeeze(-1)


class GraphSAGE(nn.Module):
    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.conv1 = SAGEConv(in_dim, hidden)
        self.conv2 = SAGEConv(hidden, hidden)
        self.head = nn.Linear(hidden, 1)

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.relu(self.conv2(x, data.edge_index))
        return self.head(x).squeeze(-1)


class GAT(nn.Module):
    def __init__(self, in_dim, hidden=64, heads=4):
        super().__init__()
        self.conv1 = GATConv(in_dim, hidden // heads, heads=heads, dropout=0.3)
        self.conv2 = GATConv(hidden, hidden // heads, heads=heads, dropout=0.3)
        self.head = nn.Linear(hidden, 1)

    def forward(self, data):
        x = F.elu(self.conv1(data.x, data.edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.elu(self.conv2(x, data.edge_index))
        return self.head(x).squeeze(-1)


class GIN(nn.Module):
    def __init__(self, in_dim, hidden=64):
        super().__init__()
        nn1 = nn.Sequential(nn.Linear(in_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        nn2 = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.conv1 = GINConv(nn1)
        self.conv2 = GINConv(nn2)
        self.head = nn.Linear(hidden, 1)

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.relu(self.conv2(x, data.edge_index))
        return self.head(x).squeeze(-1)

### 4b — Evidence-Gated GNN (EG-GNN)

The key novelty: a **learned confidence gate** on each edge that modulates
message passing based on the evidence quality of the source node.

$$m_{j \to i} = \sigma(\text{gate}(x_j)) \cdot W \cdot x_j$$

The gate learns which evidence profiles are trustworthy for message passing,
so a gene with strong multi-source evidence (high P, T, C, D, K) sends
stronger messages than a gene with only one weak signal.

In [ ]:
from torch_geometric.nn import MessagePassing
from torch_geometric.utils import add_self_loops, degree


class EvidenceGatedConv(MessagePassing):
    """Graph convolution with a learned evidence gate on messages."""

    def __init__(self, in_dim, out_dim):
        super().__init__(aggr='add')
        self.lin = nn.Linear(in_dim, out_dim, bias=False)
        self.gate_net = nn.Sequential(
            nn.Linear(in_dim, in_dim),
            nn.ReLU(),
            nn.Linear(in_dim, 1),
            nn.Sigmoid()
        )
        self.bias = nn.Parameter(torch.zeros(out_dim))

    def forward(self, x, edge_index, edge_weight=None):
        # Compute gate values for ALL nodes
        gate = self.gate_net(x)  # (N, 1)

        # Transform features
        x_transformed = self.lin(x)  # (N, out_dim)

        # Normalisation (symmetric)
        edge_index_sl, _ = add_self_loops(edge_index, num_nodes=x.size(0))
        row, col = edge_index_sl
        deg = degree(col, x.size(0), dtype=x.dtype)
        deg_inv_sqrt = deg.pow(-0.5)
        deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
        norm = deg_inv_sqrt[row] * deg_inv_sqrt[col]

        # Propagate with gate
        out = self.propagate(edge_index_sl, x=x_transformed,
                             gate=gate, norm=norm)
        return out + self.bias

    def message(self, x_j, gate_j, norm):
        # gate_j: gate value of source node j
        return norm.view(-1, 1) * gate_j * x_j


class EvidenceGatedGNN(nn.Module):
    """Two-layer Evidence-Gated GNN for bridge gene ranking."""

    def __init__(self, in_dim, hidden=64):
        super().__init__()
        self.conv1 = EvidenceGatedConv(in_dim, hidden)
        self.conv2 = EvidenceGatedConv(hidden, hidden)
        self.head = nn.Linear(hidden, 1)

    def forward(self, data):
        x = F.relu(self.conv1(data.x, data.edge_index, data.edge_attr))
        x = F.dropout(x, p=0.3, training=self.training)
        x = F.relu(self.conv2(x, data.edge_index, data.edge_attr))
        return self.head(x).squeeze(-1)

    def get_gate_values(self, data):
        """Extract gate values for interpretability."""
        with torch.no_grad():
            g1 = self.conv1.gate_net(data.x)
            h = F.relu(self.conv1(data.x, data.edge_index, data.edge_attr))
            g2 = self.conv2.gate_net(h)
        return g1.squeeze(), g2.squeeze()


print('EvidenceGatedGNN defined.')
print('Gate mechanism: σ(MLP(x_j)) · W·x_j — learned per-node confidence.')

## 5 · Training & Evaluation Functions

In [ ]:
from sklearn.metrics import (average_precision_score, roc_auc_score,
                             ndcg_score, mean_squared_error)


def train_epoch(model, data, optimizer, criterion):
    model.train()
    optimizer.zero_grad()
    pred = model(data)
    loss = criterion(pred[data.train_mask], data.y[data.train_mask])
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def evaluate(model, data, mask):
    model.eval()
    pred = model(data)
    pred_np = pred[mask].cpu().numpy()
    true_np = data.y[mask].cpu().numpy()
    return pred_np, true_np


def compute_ranking_metrics(pred, true, top_k=50):
    """Compute ranking metrics for bridge gene prediction."""
    results = {}

    # Binary label: top 5% PCBS genes = positives
    threshold = np.percentile(true, 95)
    binary_true = (true >= threshold).astype(int)

    # Handle edge case: if no positives or all positives
    if binary_true.sum() == 0 or binary_true.sum() == len(binary_true):
        return {'AUPRC': 0, 'AUROC': 0.5, f'Prec@{top_k}': 0,
                f'Rec@{top_k}': 0, 'NDCG': 0, 'RMSE': 999}

    # AUPRC (primary)
    results['AUPRC'] = average_precision_score(binary_true, pred)

    # AUROC (complementary)
    results['AUROC'] = roc_auc_score(binary_true, pred)

    # Precision@K and Recall@K
    top_k_idx = np.argsort(pred)[::-1][:top_k]
    top_k_true = binary_true[top_k_idx]
    results[f'Prec@{top_k}'] = top_k_true.sum() / top_k
    total_pos = binary_true.sum()
    results[f'Rec@{top_k}'] = top_k_true.sum() / total_pos if total_pos > 0 else 0

    # NDCG
    results['NDCG'] = ndcg_score(true.reshape(1, -1), pred.reshape(1, -1))

    # RMSE (regression quality)
    results['RMSE'] = np.sqrt(mean_squared_error(true, pred))

    return results


def train_model(model, data, epochs=300, lr=0.005, patience=30, verbose=True):
    """Train with early stopping on validation loss."""
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = nn.MSELoss()

    best_val_loss = float('inf')
    best_state = None
    wait = 0
    history = {'train_loss': [], 'val_loss': []}

    for epoch in range(1, epochs + 1):
        train_loss = train_epoch(model, data, optimizer, criterion)

        # Validation loss
        model.eval()
        with torch.no_grad():
            pred = model(data)
            val_loss = criterion(pred[data.val_mask], data.y[data.val_mask]).item()

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            wait = 0
        else:
            wait += 1

        if wait >= patience:
            if verbose:
                print(f'  Early stop at epoch {epoch} (best val_loss: {best_val_loss:.6f})')
            break

        if verbose and epoch % 50 == 0:
            print(f'  Epoch {epoch:3d} — train: {train_loss:.6f}  val: {val_loss:.6f}')

    model.load_state_dict(best_state)
    return history


print('Training & evaluation functions defined.')

## 6 · PPR Baseline (No Learning)

In [ ]:
# Build networkx graph for PPR
G_nx = nx.Graph()
edge_idx_np = data.edge_index.cpu().numpy()
for i in range(0, edge_idx_np.shape[1], 2):  # skip reverse edges
    G_nx.add_edge(edge_idx_np[0, i], edge_idx_np[1, i])

# Personalised PageRank — personalise on known bridge genes (train set high-PCBS)
train_y = data.y[data.train_mask].cpu().numpy()
train_idx = data.train_mask.cpu().numpy().nonzero()[0]
top_train = train_idx[train_y >= np.percentile(train_y, 90)]

personal = {n: 0.0 for n in G_nx.nodes()}
for n in top_train:
    if n in personal:
        personal[n] = 1.0

if sum(personal.values()) > 0:
    ppr_scores = nx.pagerank(G_nx, alpha=0.85, personalization=personal)
else:
    ppr_scores = nx.pagerank(G_nx, alpha=0.85)

ppr_pred = np.array([ppr_scores.get(i, 0) for i in range(num_nodes)])

# Evaluate on test set
test_mask_np = data.test_mask.cpu().numpy()
ppr_metrics = compute_ranking_metrics(ppr_pred[test_mask_np],
                                       data.y.cpu().numpy()[test_mask_np])

print('PPR Baseline (test set):')
for k, v in ppr_metrics.items():
    print(f'  {k}: {v:.4f}')

## 7 · Train All GNN Models

In [ ]:
in_dim = data.num_node_features
HIDDEN = 64
EPOCHS = 300
PATIENCE = 30
NUM_RUNS = 3  # average over multiple seeds for stability

model_classes = {
    'GCN':       lambda: GCN(in_dim, HIDDEN),
    'GraphSAGE': lambda: GraphSAGE(in_dim, HIDDEN),
    'GAT':       lambda: GAT(in_dim, HIDDEN),
    'GIN':       lambda: GIN(in_dim, HIDDEN),
    'EG-GNN':    lambda: EvidenceGatedGNN(in_dim, HIDDEN),
}

all_results = {'PPR': ppr_metrics}
best_models = {}

for model_name, model_fn in model_classes.items():
    print(f'\n{"="*50}')
    print(f'Training {model_name} ({NUM_RUNS} runs)')
    print(f'{"="*50}')

    run_metrics = []
    best_run_auprc = -1

    for run in range(NUM_RUNS):
        torch.manual_seed(42 + run)
        np.random.seed(42 + run)

        model = model_fn().to(device)
        print(f'\n  Run {run+1}/{NUM_RUNS}')
        history = train_model(model, data, epochs=EPOCHS, patience=PATIENCE,
                              verbose=True)

        pred_np, true_np = evaluate(model, data, data.test_mask)
        metrics = compute_ranking_metrics(pred_np, true_np)
        run_metrics.append(metrics)

        if metrics['AUPRC'] > best_run_auprc:
            best_run_auprc = metrics['AUPRC']
            best_models[model_name] = model

        print(f'  Test AUPRC: {metrics["AUPRC"]:.4f}  AUROC: {metrics["AUROC"]:.4f}')

    # Average metrics
    avg_metrics = {}
    for key in run_metrics[0]:
        vals = [m[key] for m in run_metrics]
        avg_metrics[key] = np.mean(vals)
        avg_metrics[f'{key}_std'] = np.std(vals)

    all_results[model_name] = avg_metrics
    print(f'\n  {model_name} avg: AUPRC={avg_metrics["AUPRC"]:.4f}±{avg_metrics["AUPRC_std"]:.4f}  '
          f'AUROC={avg_metrics["AUROC"]:.4f}±{avg_metrics["AUROC_std"]:.4f}')

## 8 · Benchmark Comparison Table

In [ ]:
# Build comparison table
metric_keys = ['AUPRC', 'AUROC', 'Prec@50', 'Rec@50', 'NDCG', 'RMSE']
model_order = ['PPR', 'GCN', 'GraphSAGE', 'GAT', 'GIN', 'EG-GNN']

print('\nBenchmark Results (Test Set)')
print('=' * 85)
header = f'{"Model":<12}'
for m in metric_keys:
    header += f'{m:>12}'
print(header)
print('-' * 85)

rows = []
for model_name in model_order:
    res = all_results[model_name]
    row_str = f'{model_name:<12}'
    row_data = {'Model': model_name}
    for m in metric_keys:
        val = res.get(m, 0)
        std = res.get(f'{m}_std', 0)
        if model_name == 'PPR':  # no std for PPR
            row_str += f'{val:>12.4f}'
        else:
            row_str += f'  {val:.4f}±{std:.3f}'
        row_data[m] = val
    print(row_str)
    rows.append(row_data)

# Best model
auprc_vals = {m: all_results[m].get('AUPRC', 0) for m in model_order}
best = max(auprc_vals, key=auprc_vals.get)
print(f'\nBest model by AUPRC: {best} ({auprc_vals[best]:.4f})')

# Save
results_df = pd.DataFrame(rows)
results_df.to_csv('results/benchmark_results.csv', index=False)
print('Saved: results/benchmark_results.csv')

In [ ]:
# Benchmark bar chart
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, metric in zip(axes, ['AUPRC', 'AUROC', 'NDCG']):
    vals = [all_results[m].get(metric, 0) for m in model_order]
    stds = [all_results[m].get(f'{metric}_std', 0) for m in model_order]
    colors = ['#9E9E9E'] * 5 + ['#D32F2F']  # highlight EG-GNN
    ax.bar(model_order, vals, yerr=stds, color=colors, edgecolor='white',
           capsize=3)
    ax.set_ylabel(metric)
    ax.set_title(metric, fontweight='bold')
    ax.set_ylim(0, 1.05)
    for i, v in enumerate(vals):
        ax.text(i, v + stds[i] + 0.02, f'{v:.3f}', ha='center', fontsize=8)

plt.suptitle('Model Comparison — Bridge Gene Ranking',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/benchmark_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/benchmark_comparison.png')

## 9 · Ablation Study

Quantify the contribution of each component by:
1. **Feature ablation** — remove one component at a time from node features
2. **Gate ablation** — replace evidence gate with uniform weight (GCN-no-gate)

In [ ]:
# ---- Feature ablation: remove one component at a time ----
print('Feature Ablation — EG-GNN')
print('=' * 60)

ablation_results = []

# Full model baseline
full_metrics = all_results['EG-GNN']
ablation_results.append({
    'Ablation': 'Full model (all 5)',
    'AUPRC': full_metrics['AUPRC'],
    'AUROC': full_metrics['AUROC'],
    'Delta_AUPRC': 0.0,
})

component_names = ['P_g', 'T_g', 'C_g', 'D_g', 'K_g']
component_labels = ['P(g) PMOS', 'T(g) Transition', 'C(g) Cancer',
                     'D(g) Directional', 'K(g) Independent']

for drop_idx, (comp, label) in enumerate(zip(component_names, component_labels)):
    # Create ablated features (zero out one component)
    x_ablated = data.x.clone()
    x_ablated[:, drop_idx] = 0.0

    data_abl = Data(x=x_ablated, edge_index=data.edge_index,
                    edge_attr=data.edge_attr, y=data.y,
                    train_mask=data.train_mask, val_mask=data.val_mask,
                    test_mask=data.test_mask).to(device)

    # Train ablated model
    torch.manual_seed(42)
    model_abl = EvidenceGatedGNN(in_dim, HIDDEN).to(device)
    train_model(model_abl, data_abl, epochs=EPOCHS, patience=PATIENCE, verbose=False)

    pred_np, true_np = evaluate(model_abl, data_abl, data_abl.test_mask)
    metrics = compute_ranking_metrics(pred_np, true_np)

    delta = metrics['AUPRC'] - full_metrics['AUPRC']
    ablation_results.append({
        'Ablation': f'Without {label}',
        'AUPRC': metrics['AUPRC'],
        'AUROC': metrics['AUROC'],
        'Delta_AUPRC': delta,
    })
    print(f'  Without {label:<18}: AUPRC={metrics["AUPRC"]:.4f}  '
          f'Δ={delta:+.4f}  AUROC={metrics["AUROC"]:.4f}')

# ---- Gate ablation: EG-GNN without gate (equivalent to GCN with same arch) ----
print(f'\n  GCN (no gate) reference:  AUPRC={all_results["GCN"]["AUPRC"]:.4f}')
gate_delta = full_metrics['AUPRC'] - all_results['GCN']['AUPRC']
print(f'  Gate contribution:  Δ AUPRC = {gate_delta:+.4f}')

ablation_results.append({
    'Ablation': 'Without gate (= GCN)',
    'AUPRC': all_results['GCN']['AUPRC'],
    'AUROC': all_results['GCN']['AUROC'],
    'Delta_AUPRC': -gate_delta,
})

abl_df = pd.DataFrame(ablation_results)
abl_df.to_csv('results/ablation_results.csv', index=False)
print(f'\nSaved: results/ablation_results.csv')

In [ ]:
# Ablation waterfall chart
fig, ax = plt.subplots(figsize=(10, 6))

labels = [r['Ablation'] for r in ablation_results]
deltas = [r['Delta_AUPRC'] for r in ablation_results]
auprc_vals = [r['AUPRC'] for r in ablation_results]

colors = ['#1565C0' if d == 0 else '#D32F2F' if d < 0 else '#4CAF50'
          for d in deltas]

y_pos = range(len(labels)-1, -1, -1)
ax.barh(list(y_pos), auprc_vals, color=colors, edgecolor='white')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(labels)
ax.set_xlabel('AUPRC')
ax.set_title('Ablation Study — Component Importance', fontweight='bold')

for i, (yp, v, d) in enumerate(zip(y_pos, auprc_vals, deltas)):
    label = f'{v:.4f}'
    if d != 0:
        label += f' ({d:+.4f})'
    ax.text(v + 0.005, yp, label, va='center', fontsize=9)

plt.tight_layout()
plt.savefig('results/ablation_study.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/ablation_study.png')

## 10 · Evidence Gate Interpretability

Examine what the gate learned — which genes get high vs low gate values,
and how gate values relate to evidence depth.

In [ ]:
eg_model = best_models['EG-GNN']
g1, g2 = eg_model.get_gate_values(data)
gate1_np = g1.cpu().numpy()
gate2_np = g2.cpu().numpy()

# Add to node info
gate_df = pd.DataFrame({
    'Gene': all_genes,
    'Gate_L1': gate1_np,
    'Gate_L2': gate2_np,
    'PCBS': data.y.cpu().numpy(),
    'n_components': data.x.cpu().numpy().sum(axis=1) > 0,
})
gate_df['n_comp'] = (data.x.cpu().numpy() > 0).sum(axis=1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Gate L1 vs PCBS
axes[0].scatter(gate_df['Gate_L1'], gate_df['PCBS'], s=10, alpha=0.4, color='#1565C0')
axes[0].set_xlabel('Gate Value (Layer 1)')
axes[0].set_ylabel('PCBS Score')
axes[0].set_title('Layer 1 Gate vs PCBS', fontweight='bold')

# Gate by n_components
for nc in sorted(gate_df['n_comp'].unique()):
    vals = gate_df[gate_df['n_comp'] == nc]['Gate_L1']
    if len(vals) > 5:
        axes[1].boxplot(vals, positions=[nc], widths=0.6)
axes[1].set_xlabel('Number of Evidence Components')
axes[1].set_ylabel('Gate Value (Layer 1)')
axes[1].set_title('Gate Value by Evidence Depth', fontweight='bold')

# Gate distribution
axes[2].hist(gate1_np, bins=50, color='#1565C0', alpha=0.7, label='Layer 1')
axes[2].hist(gate2_np, bins=50, color='#D32F2F', alpha=0.5, label='Layer 2')
axes[2].set_xlabel('Gate Value')
axes[2].set_ylabel('Count')
axes[2].set_title('Gate Value Distributions', fontweight='bold')
axes[2].legend()

plt.suptitle('Evidence Gate Interpretability', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('results/gate_interpretability.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/gate_interpretability.png')

## 11 · Top Predicted Bridge Genes

In [ ]:
# Get predictions from best EG-GNN
eg_model.eval()
with torch.no_grad():
    all_pred = eg_model(data).cpu().numpy()

pred_df = pd.DataFrame({
    'Gene': all_genes,
    'PCBS_true': data.y.cpu().numpy(),
    'PCBS_pred': all_pred,
})

# Merge with node features
pred_df = pred_df.merge(nodes_df[['Gene', 'Degree', 'n_components']], on='Gene', how='left')
pred_df['Rank'] = pred_df['PCBS_pred'].rank(ascending=False, method='min').astype(int)
pred_df = pred_df.sort_values('PCBS_pred', ascending=False)

print('Top 30 Predicted Bridge Genes (EG-GNN)')
print('=' * 75)
print(f'{"Rank":<6} {"Gene":<10} {"Pred":>8} {"True":>8} {"Degree":>7} {"#Comp":>6}')
print('-' * 75)
for _, r in pred_df.head(30).iterrows():
    print(f'{int(r["Rank"]):<6} {r["Gene"]:<10} {r["PCBS_pred"]:8.4f} '
          f'{r["PCBS_true"]:8.4f} {int(r.get("Degree", 0)):7d} '
          f'{int(r.get("n_components", 0)):6d}')

# Save full predictions
pred_df.to_csv('results/gnn_predictions.csv', index=False)
print(f'\nSaved: results/gnn_predictions.csv ({len(pred_df):,} genes)')

## 12 · Predicted vs True PCBS

In [ ]:
from scipy import stats

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Scatter — all genes
rho, pval = stats.spearmanr(pred_df['PCBS_true'], pred_df['PCBS_pred'])
axes[0].scatter(pred_df['PCBS_true'], pred_df['PCBS_pred'],
                s=10, alpha=0.3, color='#1565C0')
axes[0].plot([0, pred_df['PCBS_true'].max()],
             [0, pred_df['PCBS_true'].max()],
             'r--', alpha=0.5, label='Perfect')
axes[0].set_xlabel('True PCBS')
axes[0].set_ylabel('Predicted PCBS')
axes[0].set_title(f'Predicted vs True PCBS\nSpearman ρ = {rho:.4f} (p = {pval:.2e})',
                  fontweight='bold')
axes[0].legend()

# Known gene check
known_genes = ['AKT1', 'PIK3CA', 'PTEN', 'ESR1', 'AR', 'INSR', 'IGF1R',
               'CYP19A1', 'CYP11A1', 'TP53', 'VEGFA', 'IL6', 'TNF',
               'ERBB2', 'MYC', 'CCND1', 'FOXO1']

known_in_pred = pred_df[pred_df['Gene'].isin(known_genes)].copy()
known_in_pred = known_in_pred.sort_values('PCBS_pred', ascending=False)

if len(known_in_pred) > 0:
    y_pos = range(len(known_in_pred))
    axes[1].barh(list(y_pos), known_in_pred['PCBS_pred'], color='#D32F2F',
                 alpha=0.7, label='Predicted')
    axes[1].barh(list(y_pos), known_in_pred['PCBS_true'], color='#1565C0',
                 alpha=0.4, label='True')
    axes[1].set_yticks(list(y_pos))
    axes[1].set_yticklabels(known_in_pred['Gene'])
    axes[1].set_xlabel('PCBS Score')
    axes[1].set_title('Known Bridge Genes — Predicted vs True', fontweight='bold')
    axes[1].legend()

plt.tight_layout()
plt.savefig('results/pred_vs_true.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/pred_vs_true.png')

## 13 · Leave-One-Source-Out Stability

In [ ]:
# Remove one data source at a time from features AND recompute target
print('Leave-One-Source-Out Stability Test')
print('=' * 60)

loso_results = []

for drop_idx, (comp, label) in enumerate(zip(component_names, component_labels)):
    # Ablated features
    x_loso = data.x.clone()
    x_loso[:, drop_idx] = 0.0

    # Recompute target: PCBS without this component (re-weight remaining)
    remaining = [i for i in range(5) if i != drop_idx]
    y_loso = x_loso[:, remaining].mean(dim=1)  # equal weight on 4 remaining

    data_loso = Data(x=x_loso, edge_index=data.edge_index,
                     edge_attr=data.edge_attr, y=y_loso,
                     train_mask=data.train_mask, val_mask=data.val_mask,
                     test_mask=data.test_mask).to(device)

    torch.manual_seed(42)
    model_loso = EvidenceGatedGNN(in_dim, HIDDEN).to(device)
    train_model(model_loso, data_loso, epochs=EPOCHS, patience=PATIENCE, verbose=False)

    # Get predictions and compare ranking to full model
    model_loso.eval()
    with torch.no_grad():
        pred_loso = model_loso(data_loso).cpu().numpy()

    # Rank correlation with full-model predictions
    rho, _ = stats.spearmanr(all_pred, pred_loso)

    # Top-50 overlap
    full_top50 = set(np.argsort(all_pred)[::-1][:50])
    loso_top50 = set(np.argsort(pred_loso)[::-1][:50])
    overlap = len(full_top50 & loso_top50)

    loso_results.append({
        'Removed': label,
        'Rank_rho': rho,
        'Top50_overlap': overlap,
    })
    print(f'  Without {label:<18}: rank ρ = {rho:.4f}, top-50 overlap = {overlap}/50')

loso_df = pd.DataFrame(loso_results)
loso_df.to_csv('results/loso_stability.csv', index=False)
print(f'\nSaved: results/loso_stability.csv')

## 14 · Training Curves

In [ ]:
# Retrain EG-GNN once more to capture history for plotting
torch.manual_seed(42)
model_plot = EvidenceGatedGNN(in_dim, HIDDEN).to(device)
history = train_model(model_plot, data, epochs=EPOCHS, patience=PATIENCE, verbose=False)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(history['train_loss'], label='Train', color='#1565C0')
ax.plot(history['val_loss'], label='Validation', color='#D32F2F')
ax.set_xlabel('Epoch')
ax.set_ylabel('MSE Loss')
ax.set_title('EG-GNN Training Curve', fontweight='bold')
ax.legend()

plt.tight_layout()
plt.savefig('results/training_curve.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results/training_curve.png')

## 15 · Save All Results

In [ ]:
# Save model state
torch.save(eg_model.state_dict(), 'results/eggnn_model.pt')
print('Saved: results/eggnn_model.pt')

# Final summary
print(f'\n{"="*55}')
print(f'NB8 — EVIDENCE-GATED GNN BENCHMARKING COMPLETE')
print(f'{"="*55}')
print(f'Graph: {data.num_nodes:,} nodes, {data.num_edges//2:,} edges')
print(f'Node features: {data.num_node_features} (P/T/C/D/K)')
print(f'\nModel rankings by AUPRC:')
sorted_models = sorted(all_results.items(),
                       key=lambda x: x[1].get('AUPRC', 0), reverse=True)
for i, (name, res) in enumerate(sorted_models, 1):
    auprc = res.get('AUPRC', 0)
    auroc = res.get('AUROC', 0)
    marker = ' ★' if name == 'EG-GNN' else ''
    print(f'  {i}. {name:<12} AUPRC={auprc:.4f}  AUROC={auroc:.4f}{marker}')

print(f'\nAblation: largest AUPRC drop from removing '
      f'{abl_df.loc[abl_df["Delta_AUPRC"].idxmin(), "Ablation"]}')

## Summary

NB8 trained an Evidence-Gated GNN on the PCBS-annotated PPI graph and
benchmarked it against five baselines (PPR, GCN, GraphSAGE, GAT, GIN).

**Key Results:**
- EG-GNN's learned evidence gate modulates message passing based on source node confidence
- Feature ablation quantifies each evidence component's contribution to ranking
- Leave-one-source-out tests ranking stability when an evidence source is removed

| Output | Description |
|--------|-------------|
| `benchmark_results.csv` | All models' AUPRC/AUROC/Prec@K/Rec@K/NDCG |
| `ablation_results.csv` | Per-component feature ablation |
| `loso_stability.csv` | Leave-one-source-out rank stability |
| `gnn_predictions.csv` | Full gene predictions from EG-GNN |
| `eggnn_model.pt` | Trained model weights |

In [ ]:
zip_path = 'results/NB8_GNN_results.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in ['results/benchmark_results.csv',
              'results/ablation_results.csv',
              'results/loso_stability.csv',
              'results/gnn_predictions.csv',
              'results/eggnn_model.pt',
              'results/benchmark_comparison.png',
              'results/ablation_study.png',
              'results/gate_interpretability.png',
              'results/pred_vs_true.png',
              'results/training_curve.png']:
        if os.path.exists(f):
            zf.write(f)

print(f'Results ZIP: {zip_path}')

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Not running in Colab — download manually.')